# Predição de Classificação Binária para o Tipo de Câncer de Mama

O câncer de mama é o câncer mais comum entre as mulheres em todo o mundo. Ele responde por 25% de todos os casos de câncer e afetou mais de 2,1 milhões de pessoas apenas em 2015. Ele começa quando as células da mama começam a crescer de forma descontrolada, tipicamente formando tumores que podem ser detectados por raio-X ou sentidos como nódulos na região mamária.

Neste projeto, vou construir um **modelo de aprendizado de máquina para classificar tumores em malignos (cancerosos) ou benignos (não cancerosos).**

Vou utilizar o seguinte **pipeline** baseado no **framework CRISP-DM:**

1. **Definir o problema de negócio.**
2. **Coletar os dados e obter uma visão geral deles.**
3. **Dividir os dados em conjuntos de treino e teste.**
4. **Explorar os dados (análise exploratória de dados).**
5. **Engenharia de features, limpeza e pré-processamento dos dados.**
6. **Treinamento, comparação, seleção de features e ajuste de modelos.**
7. **Teste e avaliação do modelo final de produção.**
8. **Concluir e interpretar os resultados do modelo.**
9. **Implantação (Deploy).**

Neste **notebook**, realizarei a **análise exploratória de dados (EDA), cobrindo as etapas de 1 a 4 do pipeline acima.** O principal **objetivo** é **descobrir insights** que forneçam **informações valiosas sobre os padrões dos tumores** dentro das features disponíveis. Além disso, abordarei essas etapas detalhadamente a seguir, explicando a justificativa por trás de cada decisão.

---

## 1. Definir o Problema de Negócio

O diagnóstico do câncer de mama depende fortemente da identificação precisa e precoce de tumores malignos para prevenir a metástase e melhorar as taxas de sobrevivência dos pacientes. Na prática clínica, lesões suspeitas identificadas por meio de exames de rastreamento (como a mamografia) são frequentemente investigadas mais a fundo utilizando a **Punção Aspirativa por Agulha Fina (PAAF / Fine Needle Aspiration - FNA)** de uma massa mamária.

O conjunto de dados utilizado neste projeto origina-se de imagens digitalizadas de lâminas de PAAF. Um software de imagem especializado processa essas lâminas para computar características quantitativas que descritivamente representam os núcleos celulares presentes na imagem. Esse processo gera as features encontradas em nosso dataset — tais como `radius_mean`, `texture_se` e `perimeter_worst` —, que quantificam matematicamente propriedades espaciais (raio, perímetro, área), variações morfológicas (textura, suavidade, concavidade, simetria) e suas respectivas métricas de variação (média, erro padrão e os "piores" ou maiores valores). Traduzir essas medições celulares em um sistema preditivo automatizado ajuda a reduzir a subjetividade humana nos procedimentos diagnósticos.

### Qual é o contexto
* O câncer de mama representa um dos desafios de saúde mais prevalentes e críticos globalmente, exigindo métodos de detecção rápidos e confiáveis.
* O diagnóstico tradicional depende fortemente da análise microscópica manual de amostras de tecido, o que pode ser demorado e estar sujeito à variabilidade entre observadores.
* O dataset faz a ponte entre a imagem médica e a ciência de dados, extraindo métricas celulares quantitativas de procedimentos de Punção Aspirativa por Agulha Fina (PAAF).

### Quais são os objetivos do projeto
* Construir um modelo robusto de classificação binária para diferenciar com precisão entre tumores de mama malignos (cancerosos) e benignos (não cancerosos).
* Minimizar erros críticos de classificação, reduzindo especificamente os falsos negativos (falhar em detectar a malignidade) para evitar atrasos nos tratamentos.
* Identificar quais features morfológicas (ex.: raio, perímetro, textura) possuem o maior poder preditivo para a classificação do câncer.

### Por que a abordagem de ordenação por probabilidade/escore
* Fornece um escore contínuo de probabilidade de malignidade em vez de apenas um rótulo binário rígido, oferecendo uma avaliação de risco mais refinada.
* Permite que profissionais médicos ajustem os limiares de decisão com base na tolerância ao risco clínico (ex.: priorizar alta sensibilidade para contextos de rastreamento).
* Possibilita a ordenação de pacientes ou amostras de tecidos por seu escore de risco, otimizando a alocação de recursos para revisões secundárias urgentes.

### Quais são os benefícios
* Fornece uma segunda opinião confiável e baseada em dados para auxiliar patologistas e clínicos na tomada de decisões.
* Reduz a subjetividade diagnóstica e o erro humano na avaliação da morfologia celular a partir de imagens de lâminas.
* Agiliza o fluxo de trabalho clínico, acelerando o caminho desde o rastreamento inicial até o cuidado definitivo do paciente e o planejamento do tratamento.

### Importing libraries

In [1]:
# Data manipulation and visualization.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Split the data.
from sklearn.model_selection import train_test_split

# Utils


# Filter warnings.
import warnings
warnings.filterwarnings('ignore')

### Visualization Settings
%matplotlib inline

mpl.style.use('ggplot') 

mpl.rcParams['axes.facecolor']      = 'white'
mpl.rcParams['axes.linewidth']      = 1
mpl.rcParams['xtick.color']         = 'black'
mpl.rcParams['ytick.color']         = 'black'
mpl.rcParams['grid.color']          = 'lightgray'
mpl.rcParams['figure.dpi']          = 150
mpl.rcParams['axes.grid']           = True
mpl.rcParams['font.size']           = 12

# Palette Setting
color_palette = ['#023047', '#e85d04', '#0077b6', '#ff8200', '#0096c7', '#ff9c33']
# Setting as the palette
sns.set_palette(sns.color_palette(color_palette))
# Display
sns.color_palette(color_palette)

[(0.00784313725490196, 0.18823529411764706, 0.2784313725490196),
 (0.9098039215686274, 0.36470588235294116, 0.01568627450980392),
 (0.0, 0.4666666666666667, 0.7137254901960784),
 (1.0, 0.5098039215686274, 0.0),
 (0.0, 0.5882352941176471, 0.7803921568627451),
 (1.0, 0.611764705882353, 0.2)]

### 2. Entendendo os Dados

* O conjunto de dados foi obtido no Kaggle: [Breast Cancer Dataset](https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset/data).
* A origem dos dados remete a lâminas digitalizadas de Punção Aspirativa por Agulha Fina (PAAF / FNA) de massas mamárias.
* Contém dados de pacientes com suspeita de câncer de mama que realizaram PAAF/FNA, trazendo informações sobre medidas espaciais e geométricas do núcleo das células coletadas, além de variações morfométricas e seus respectivos sufixos descritivos.
* Os dados já vêm rotulados, com tumores malignos identificados como 1 e benignos identificados como 0 na variável alvo.